<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA2/IL2.1/4-crewai-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. Orquestación de Agentes con CrewAI

## Objetivos de Aprendizaje
- Entender el concepto de agentes colaborativos y los roles en un "equipo" (Crew).
- Aprender los componentes clave de CrewAI: `Agent`, `Task`, `Tool` y `Crew`.
- Definir un equipo de agentes (un investigador y un escritor) para realizar una tarea compleja.
- Ejecutar el `Crew` y observar cómo los agentes colaboran y se pasan el trabajo entre ellos.
- **Comprender cómo se configura el proveedor de LLM en CrewAI, que usa LiteLLM y no LangChain**.

## ¿Qué es CrewAI y por qué usarlo?

Mientras que LangChain proporciona los bloques de construcción fundamentales para crear un agente, **CrewAI** se especializa en la **orquestación de múltiples agentes autónomos**. La idea central es que, para resolver tareas complejas, es más eficiente tener un equipo de agentes especializados que colaboren, en lugar de un solo agente que intente hacerlo todo.

**Analogía:** Piensa en una agencia de marketing. No tienes una sola persona que es experta en investigación, redacción, diseño y redes sociales. Tienes un equipo donde cada miembro tiene un rol claro. CrewAI aplica este concepto a los agentes de IA.

**Ventajas de CrewAI:**
- **Roles Especializados**: Permite definir agentes con roles, objetivos (`goal`) e historias de fondo (`backstory`) específicas, lo que los hace más efectivos en su nicho.
- **Colaboración Autónoma**: Los agentes pueden delegar tareas entre ellos de forma autónoma.
- **Procesos Secuenciales y Jerárquicos**: Soporta flujos de trabajo donde las tareas se completan en un orden específico.
- **Claridad y Estructura**: El código es muy declarativo y fácil de leer, ya que se centra en definir el equipo y sus responsabilidades.

## Relación entre CrewAI y LangChain

CrewAI **utiliza LangChain internamente** para manejar los LLMs y las herramientas. Esto significa que:

1. **CrewAI** se encarga de la orquestación de agentes (el "director de orquesta")
2. **LangChain** proporciona la interfaz con los modelos y herramientas (los "instrumentos")

Esta relación requiere configuraciones específicas que veremos en este notebook.

### 1. Instalación y Configuración

### 2. Configuración del proveedor de LLM

**⚠️ IMPORTANTE:** Esta es la parte crítica que causa problemas si no se configura correctamente.

**El Problema:**
- CrewAI utiliza LangChain internamente
- LangChain busca las variables de entorno `OPENAI_API_KEY` y `OPENAI_API_BASE`
- Nosotros tenemos `GITHUB_TOKEN` y `OPENAI_BASE_URL`
- Necesitamos "mapear" nuestras variables a las que LangChain espera

**La Solución:**
Comprobamos que las credenciales llegaron. CrewAI no lee LangChain: recibe el proveedor explícitamente en su clase `LLM`.

In [ ]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q crewai python-dotenv wikipedia


> ⏱️ **En Google Colab, la primera celda tarda varios minutos.** CrewAI arrastra
> un árbol de dependencias grande (unos 140 paquetes). Espera a que termine antes
> de ejecutar el resto.
>
> Si al importar `crewai` aparece un error de versiones, ve a
> `Entorno de ejecución → Reiniciar sesión` y vuelve a ejecutar desde la primera
> celda: Colab trae algunas librerías precargadas y a veces hay que reiniciar
> para que tome las que acaba de instalar.


In [ ]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Verificar que las credenciales llegaron
print("🔍 Verificando configuración:")
print(f"LLM_BASE_URL: {os.environ.get('LLM_BASE_URL', '❌ No configurado')}")
print(f"LLM_MODEL   : {os.environ.get('LLM_MODEL', '❌ No configurado')}")
print(f"LLM_API_KEY : {'✅ Configurado' if os.environ.get('LLM_API_KEY') else '❌ No configurado'}")

🔍 Verificando configuración:
LLM_BASE_URL: https://api.mistral.ai/v1
LLM_MODEL   : mistral-small-latest
LLM_API_KEY : ✅ Configurado


### 3. Configuración del LLM

Ahora configuramos el LLM usando LangChain. Gracias a la configuración anterior, LangChain automáticamente usará nuestras variables de entorno mapeadas.

In [ ]:
import wikipedia
from crewai import LLM

# Configurar el idioma de Wikipedia
wikipedia.set_lang('es')

# 🧠 Configurar el LLM para CrewAI
# CrewAI usa LiteLLM por debajo, no LangChain. El prefijo "openai/" le indica
# que hable el protocolo de OpenAI contra el `base_url` que le damos, lo que
# permite usar cualquier proveedor compatible (Groq, Mistral, etc.).
try:
    llm = LLM(
        model="openai/" + os.getenv("LLM_MODEL", "mistral-small-latest"),
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        temperature=0,
    )

    # Probar que el LLM funciona
    print("✅ LLM configurado exitosamente.")
    print(f"📝 Respuesta de prueba: {llm.call('Responde solo: funcionando')[:60]}")

except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    print("💡 Verifica que LLM_BASE_URL y LLM_API_KEY estén configurados correctamente.")
    llm = None

✅ LLM configurado exitosamente.


📝 Respuesta de prueba: Funcionando


### 4. Definición de Herramientas con CrewAI

**⚠️ IMPORTANTE:** CrewAI requiere un enfoque específico para las herramientas.

**El Problema:**
- LangChain usa el decorador `@tool` para definir herramientas
- CrewAI requiere que las herramientas hereden de `BaseTool`
- Mezclar ambos enfoques causa errores

**La Solución:**
Usar `BaseTool`, que en CrewAI 1.x se importa desde `crewai.tools` (antes estaba en `crewai_tools`).

In [ ]:
# En CrewAI 1.x, BaseTool vive en `crewai.tools` (antes estaba en `crewai_tools`)
from crewai.tools import BaseTool

# 🔧 Definimos la herramienta como subclase de BaseTool
class WikipediaSearchTool(BaseTool):
    name: str = "Wikipedia Search Tool"
    description: str = "Busca en Wikipedia un tema y devuelve un resumen detallado. Es ideal para obtener información sobre personas, lugares, conceptos históricos y científicos."

    def _run(self, query: str) -> str:
        """Ejecuta la búsqueda en Wikipedia"""
        try:
            # Configurar el idioma de Wikipedia
            wikipedia.set_lang("es")
            # Devolver un resumen detallado para que el escritor tenga más material
            return wikipedia.summary(query, sentences=5)
        except wikipedia.exceptions.PageError:
            return f"No se encontró ninguna página para '{query}'. Intenta con un término más específico."
        except wikipedia.exceptions.DisambiguationError as e:
            return f"La búsqueda para '{query}' es ambigua. Opciones disponibles: {e.options[:3]}. Especifica cuál te interesa."
        except Exception as e:
            return f"Error al buscar en Wikipedia: {str(e)}"

# Crear la instancia de la herramienta
wikipedia_tool = WikipediaSearchTool()
tools = [wikipedia_tool]

# Probar la herramienta
try:
    test_result = wikipedia_tool._run("Albert Einstein")
    print("✅ Herramienta de Wikipedia configurada y probada exitosamente.")
    print(f"📝 Resultado de prueba: {test_result[:100]}...")
except Exception as e:
    print(f"❌ Error probando la herramienta: {e}")

print(f"\n🔧 {len(tools)} herramienta(s) disponible(s) para los agentes.")

✅ Herramienta de Wikipedia configurada y probada exitosamente.
📝 Resultado de prueba: Albert Einstein pronunciación en alemán: /ˈalbɐt ˈaɪnʃtaɪn/ ();​ (Ulm, 14 de marzo de 1879-Princeton...

🔧 1 herramienta(s) disponible(s) para los agentes.


### 5. Creación del Equipo de Agentes (Crew)

Ahora definimos nuestro equipo de agentes especializados:

1. **Investigador (Researcher)**: Busca información detallada usando Wikipedia
2. **Escritor (Writer)**: Transforma la información en una biografía bien redactada

**⚠️ IMPORTANTE:** Otro error común es el parámetro `verbose` en `Crew`.

In [ ]:
from crewai import Agent, Task, Crew, Process

# 🕵️ Agente 1: El Investigador
researcher = Agent(
    role="Investigador Senior",
    goal="Encontrar información completa y precisa sobre personas históricas, científicos y figuras importantes utilizando fuentes confiables.",
    backstory="""Eres un investigador académico con años de experiencia en la búsqueda de información histórica y científica.
    Tu especialidad es encontrar datos precisos y relevantes en Wikipedia y otras fuentes confiables.
    Te enorgulleces de la exactitud de tu trabajo y siempre proporcionas contexto histórico relevante.
    No escribes biografías completas, tu trabajo es recopilar los datos más importantes y precisos.""",
    tools=tools,
    llm=llm,
    verbose=True,
    allow_delegation=False  # Este agente no delega trabajo
)

# ✍️ Agente 2: El Escritor
writer = Agent(
    role="Escritor de Biografías",
    goal="Crear biografías atractivas, bien estructuradas y fáciles de leer basadas en la información proporcionada por el investigador.",
    backstory="""Eres un escritor profesional especializado en biografías y divulgación científica.
    Tu habilidad única es transformar datos técnicos y históricos en narrativas cautivadoras que son
    tanto informativas como accesibles para el público general.
    Tienes un don especial para destacar los aspectos más interesantes de la vida de las personas
    y presentar sus logros de manera inspiradora.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Agentes creados exitosamente:")
print(f"🕵️ {researcher.role}")
print(f"✍️ {writer.role}")

✅ Agentes creados exitosamente:
🕵️ Investigador Senior
✍️ Escritor de Biografías


### 6. Definición de Tareas

Las tareas definen exactamente qué debe hacer cada agente y cómo se relacionan entre ellas.

In [ ]:
# 🔍 Tarea 1: Investigación
research_task = Task(
    description="""Busca información detallada sobre Marie Curie en Wikipedia.
    Enfócate en:
    - Sus descubrimientos científicos más importantes
    - Su impacto en la ciencia y la sociedad
    - Datos biográficos clave (fechas, lugares, educación)
    - Sus premios y reconocimientos
    - Su legado científico

    Proporciona información precisa y bien organizada que el escritor pueda usar.""",
    expected_output="Un resumen detallado de 4-6 párrafos con los datos más importantes sobre la vida, descubrimientos y legado de Marie Curie.",
    agent=researcher
)

# ✍️ Tarea 2: Escritura
write_task = Task(
    description="""Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.

    Requisitos:
    - Mínimo 5 párrafos bien estructurados
    - Estilo atractivo y accesible
    - Incluir sus logros más importantes
    - Destacar su impacto en la ciencia
    - Formato Markdown con encabezados apropiados
    - Tono inspirador pero preciso

    La biografía debe ser educativa e inspiradora para lectores de todas las edades.""",
    expected_output="Una biografía completa en formato Markdown, bien estructurada y atractiva, de al menos 5 párrafos.",
    agent=writer,
    context=[research_task]  # Esta tarea depende del resultado de la investigación
)

print("✅ Tareas definidas exitosamente:")
print(f"🔍 Tarea de investigación: {research_task.description[:50]}...")
print(f"✍️ Tarea de escritura: {write_task.description[:50]}...")

✅ Tareas definidas exitosamente:
🔍 Tarea de investigación: Busca información detallada sobre Marie Curie en W...
✍️ Tarea de escritura: Usando la información recopilada por el investigad...


### 7. Ensamblaje del Equipo (Crew)

**⚠️ CONFIGURACIÓN CORREGIDA:** El parámetro `verbose` debe ser boolean, no entero.

In [ ]:
# 🎯 CONFIGURACIÓN CORREGIDA: verbose debe ser boolean
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,  # Las tareas se ejecutan en orden
    verbose=True  # ✅ CORRECTO: boolean, no entero (verbose=2 causaría error)
)

print("✅ Equipo (Crew) ensamblado exitosamente:")
print(f"👥 {len(crew.agents)} agentes en el equipo")
print(f"📋 {len(crew.tasks)} tareas definidas")
print(f"🔄 Proceso: {crew.process}")
print(f"🔊 Verbose: {crew.verbose}")

✅ Equipo (Crew) ensamblado exitosamente:
👥 2 agentes en el equipo
📋 2 tareas definidas
🔄 Proceso: Process.sequential
🔊 Verbose: True


### 8. Ejecución del Crew

¡Ahora viene la magia! Ejecutamos el crew y observamos cómo los agentes colaboran.

In [ ]:
# 🚀 Ejecutar el crew
if llm is None:
    print("❌ No se puede ejecutar el crew sin un LLM configurado.")
    print("💡 Verifica la configuración de las variables de entorno en las celdas anteriores.")
else:
    try:
        print("🚀 Iniciando ejecución del crew...")
        print("📝 Observa cómo los agentes colaboran paso a paso:\n")

        # En un notebook (Jupyter o Colab) ya hay un event loop corriendo, así que
        # `crew.kickoff()` falla con "invoked synchronously from within a running
        # event loop". La variante asíncrona es la que funciona aquí; en un script
        # .py normal se usa `crew.kickoff()` directamente.
        result = await crew.kickoff_async()

        print("\n" + "="*80)
        print("🏁 RESULTADO FINAL DEL CREW")
        print("="*80)
        print(result)

    except Exception as e:
        print(f"❌ Error durante la ejecución del crew: {e}")
        print("\n🔧 Posibles soluciones:")
        print("1. Verifica que LLM_API_KEY esté configurado correctamente")
        print("2. Asegúrate de que LLM_BASE_URL esté configurado")
        print("3. Confirma que LLM_MODEL exista en el proveedor configurado")
        print("4. Verifica que no hayas mezclado @tool con BaseTool")
        print("5. Asegúrate de que verbose=True (no verbose=2)")

        # Mostrar información de debugging
        print("\n🐛 Información de debugging:")
        import traceback
        traceback.print_exc()

🚀 Iniciando ejecución del crew...
📝 Observa cómo los agentes colaboran paso a paso:



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.15                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bfd57d5f-5a6b-4979-a49d-442ac130a5dc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│  ID: 9127a4f5-a862-4690-ac1b-14197c13990a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador Senior                                                                                     │
│                                                                                                                 │
│  Task: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Escritor de Biografías                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # **Marie Curie: La Dama de la Radiactividad que Cambió la Ciencia para Siempre**                              │
│                                                                                                                 │
│  ## **Una Niña Polaca con un Sueño Inquebrantable**                                                             │
│  Marie Skłodowska-Curie nació el 7 de noviembre de 1867 en Varsovia, entonces bajo el dominio del Imperio       │
│  ruso. En una época en que las mujeres tenían vetado el acceso a la educación superior en Polonia, Marie        │
│  demostró desde pequeña una inteligencia excepcional y una sed insaciable de conocimiento. Tras graduarse con   │
│  honores en un liceo ruso —impuesto por las autoridades—, se vio obligada a trabajar como institutriz para      │
│  financiar los estudios de su hermana mayor en París. Pero su determinación no tenía límites: mientras          │
│  enseñaba en casas ajenas, asistía en secreto a la *universidad flotante* de Varsovia, una institución          │
│  clandestina donde las mujeres polacas podían estudiar ciencias. En 1891, con solo 24 años y ahorrando cada     │
│  centavo, Marie emprendió el viaje que cambiaría su vida: se trasladó a París para estudiar en la Sorbona,      │
│  donde, entre aulas abarrotadas y noches de estudio bajo la tenue luz de una lámpara, se graduó como la mejor   │
│  de su promoción en Física y Matemáticas.                                                                       │
│                                                                                                                 │
│  ## **El Amor, la Ciencia y un Descubrimiento Revolucionario**                                                  │
│  En la capital francesa, Marie conoció a Pierre Curie, un físico brillante y de carácter tímido que compartía   │
│  su pasión por la ciencia. Se casaron en 1895, formando una de las parejas más extraordinarias de la historia.  │
│  Juntos, se embarcaron en una investigación que desafiaría los cimientos de la ciencia conocida: el estudio de  │
│  los *rayos uránicos* descubiertos por Henri Becquerel. Marie, con una tenacidad inquebrantable, aisló dos      │
│  elementos hasta entonces desconocidos: el **polonio** (bautizado en honor a su amada Polonia) y el **radio**,  │
│  este último mil veces más radiactivo que el uranio. En 1903, su trabajo fue reconocido con el **Premio Nobel   │
│  de Física**, compartido con Pierre y Becquerel, convirtiéndola en la **primera mujer en recibir un Nobel**.    │
│  Pero Marie no se detuvo ahí: en 1911, ganó un segundo Nobel, esta vez en **Química**, por sus contribuciones   │
│  al avance de la química gracias al descubrimiento de estos elementos. Era la primera persona —y hasta hoy, la  │
│  única— en ganar dos Nobel en disciplinas distintas.                                                            │
│                                                                                                                 │
│  ## **De la Teoría a la Práctica: La Radiactividad que Salvó Vidas**                                            │
│  Marie Curie no solo descubrió nuevos elementos; **inventó la radiactividad como campo de estudio**. Su tesis   │
│  doctoral, *Investigaciones sobre las sustancias radiac

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.  │
│                                                                                                                 │
│      Requisitos:                                                                                                │
│      - Mínimo 5 párrafos bien estructurados                                                                     │
│      - Estilo atractivo y accesible                                                                             │
│      - Incluir sus logros más importantes                                                                       │
│      - Destacar su impacto en la ciencia                                                                        │
│      - Formato Markdown con encabezados apropiados                                                              │
│      - Tono inspirador pero preciso                                                                             │
│                                                                                                                 │
│      La biografía debe ser educativa e inspiradora para lectores de todas las edades.                           │
│  Agent: Escritor de Biografías                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bfd57d5f-5a6b-4979-a49d-442ac130a5dc                                                                       │
│  Final Output: ```markdown                                                                                      │
│  # **Marie Curie: La Dama de la Radiactividad que Cambió la Ciencia para Siempre**                              │
│                                                                                                                 │
│  ## **Una Niña Polaca con un Sueño Inquebrantable**                                                             │
│  Marie Skłodowska-Curie nació el 7 de noviembre de 1867 en Varsovia, entonces bajo el dominio del Imperio       │
│  ruso. En una época en que las mujeres tenían vetado el acceso a la educación superior en Polonia, Marie        │
│  demostró desde pequeña una inteligencia excepcional y una sed insaciable de conocimiento. Tras graduarse con   │
│  honores en un liceo ruso —impuesto por las autoridades—, se vio obligada a trabajar como institutriz para      │
│  financiar los estudios de su hermana mayor en París. Pero su determinación no tenía límites: mientras          │
│  enseñaba en casas ajenas, asistía en secreto a la *universidad flotante* de Varsovia, una institución          │
│  clandestina donde las mujeres polacas podían estudiar ciencias. En 1891, con solo 24 años y ahorrando cada     │
│  centavo, Marie emprendió el viaje que cambiaría su vida: se trasladó a París para estudiar en la Sorbona,      │
│  donde, entre aulas abarrotadas y noches de estudio bajo la tenue luz de una lámpara, se graduó como la mejor   │
│  de su promoción en Física y Matemáticas.                                                                       │
│                                                                                                                 │
│  ## **El Amor, la Ciencia y un Descubrimiento Revolucionario**                                                  │
│  En la capital francesa, Marie conoció a Pierre Curie, un físico brillante y de carácter tímido que compartía   │
│  su pasión por la ciencia. Se casaron en 1895, formando una de las parejas más extraordinarias de la historia.  │
│  Juntos, se embarcaron en una investigación que desafiaría los cimientos de la ciencia conocida: el estudio de  │
│  los *rayos uránicos* descubiertos por Henri Becquerel. Marie, con una tenacidad inquebrantable, aisló dos      │
│  elementos hasta entonces desconocidos: el **polonio** (bautizado en honor a su amada Polonia) y el **radio**,  │
│  este último mil veces más radiactivo que el uranio. En 1903, su trabajo fue reconocido con el **Premio Nobel   │
│  de Física**, compartido con Pierre y Becquerel, convirtiéndola en la **primera mujer en recibir un Nobel**.    │
│  Pero Marie no se detuvo ahí: en 1911, ganó un segundo Nobel, esta vez en **Química**, por sus contribuciones   │
│  al avance de la química gracias al descubrimiento de estos elementos. Era la primera persona —y hasta hoy, la  │
│  única— en ganar dos Nobel en disciplinas distintas.                                                            │
│                                                                                                                 │
│  ## **De la Teoría a la Práctica: La Radiactividad que Salvó Vidas**                                            │
│  Marie Curie no solo descubrió nuevos elementos; **inventó la radiactividad como campo de estudio**. Su tesis   │
│  doctoral, *Investigaciones sobre las sustancias radia


🏁 RESULTADO FINAL DEL CREW
```markdown
# **Marie Curie: La Dama de la Radiactividad que Cambió la Ciencia para Siempre**

## **Una Niña Polaca con un Sueño Inquebrantable**
Marie Skłodowska-Curie nació el 7 de noviembre de 1867 en Varsovia, entonces bajo el dominio del Imperio ruso. En una época en que las mujeres tenían vetado el acceso a la educación superior en Polonia, Marie demostró desde pequeña una inteligencia excepcional y una sed insaciable de conocimiento. Tras graduarse con honores en un liceo ruso —impuesto por las autoridades—, se vio obligada a trabajar como institutriz para financiar los estudios de su hermana mayor en París. Pero su determinación no tenía límites: mientras enseñaba en casas ajenas, asistía en secreto a la *universidad flotante* de Varsovia, una institución clandestina donde las mujeres polacas podían estudiar ciencias. En 1891, con solo 24 años y ahorrando cada centavo, Marie emprendió el viaje que cambiaría su vida: se trasladó a París para estudiar 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 🎓 Resumen de Configuraciones Críticas

### Problemas Comunes y Sus Soluciones

| **Problema** | **Síntoma** | **Solución** |
|-------------|-------------|-------------|
| **Proveedor no configurado** | `AuthenticationError: Incorrect API key` | Construir `LLM(model="openai/…", base_url=…, api_key=…)` y pasarlo a cada `Agent(llm=llm)` |
| **Herramientas incompatibles** | `'Tool' object is not callable` | Usar `BaseTool`, que en CrewAI 1.x se importa desde `crewai.tools` |
| **Parámetro verbose** | `ValidationError: Input should be a valid boolean` | Usar `verbose=True` en lugar de `verbose=2` |
| **Ejecución en notebook** | `invoked synchronously from within a running event loop` | Usar `await crew.kickoff_async()`; Jupyter y Colab ya tienen un event loop |
| **Falta `expected_output`** | `ValidationError` al crear la `Task` | En CrewAI 1.x es obligatorio describir qué debe entregar la tarea |

### Configuración correcta del proveedor

```python
from crewai import LLM

# CrewAI usa LiteLLM por debajo, no LangChain. El prefijo "openai/" le indica
# que hable el protocolo de OpenAI contra nuestro base_url, así funciona con
# cualquier proveedor compatible.
llm = LLM(
    model="openai/" + os.getenv("LLM_MODEL", "mistral-small-latest"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    temperature=0,
)

# Las herramientas se definen como subclase de BaseTool
from crewai.tools import BaseTool

class MiTool(BaseTool):
    name: str = "Mi Herramienta"
    description: str = "Descripción de la herramienta"
    def _run(self, query: str) -> str:
        return "resultado"

# Y el equipo se arma pasando el llm a cada agente
crew = Crew(
    agents=[agent1, agent2],
    tasks=[task1, task2],
    process=Process.sequential,
    verbose=True,          # booleano, no entero
)
```

### Variables de Entorno Requeridas

En Colab van en el panel 🔑 **Secrets**; en local, en el archivo `.env` de la raíz.

```bash
LLM_BASE_URL="https://api.mistral.ai/v1"
LLM_API_KEY="tu_key_de_mistral"
LLM_MODEL="mistral-small-latest"
```


## 🚀 Conclusiones

### Lo que hemos aprendido:

1. **CrewAI vs LangChain**: CrewAI orquesta equipos de agentes, LangChain proporciona los componentes individuales.

2. **Configuración crítica**: CrewAI usa LiteLLM, así que el modelo se declara con el prefijo `openai/` y se le pasan `base_url` y `api_key`.